# Notebook 04 — Final Evaluation & Demo
Evaluate CNN baseline model with full classification metrics.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

from src.dataset import load_train_val_test_datasets, optimize_dataset
from src.evaluate import (
    load_cnn_model,
    collect_predictions,
    save_classification_report,
    save_confusion_matrix,
    save_cnn_final_result,
)
from src.config import CNN_MODEL_PATH, CNN_FINAL_RESULT_PATH

## 1. Load dataset

In [ ]:
train_ds, val_ds, test_ds, class_names = load_train_val_test_datasets()
num_classes = len(class_names)

test_ds = optimize_dataset(test_ds)

print("Number of classes:", num_classes)
print("CNN model path:", CNN_MODEL_PATH)
print("Model exists:", CNN_MODEL_PATH.exists())

## 2. Load best model

In [ ]:
model = load_cnn_model()
model.summary()

## 3. Evaluate — test loss & test accuracy

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")

save_cnn_final_result(test_loss, test_accuracy)

print("Final result path:", CNN_FINAL_RESULT_PATH)
print("Final result exists:", CNN_FINAL_RESULT_PATH.exists())

## 4. Collect y_true and y_pred

In [ ]:
y_true, y_pred = collect_predictions(model, test_ds)

print("y_true shape:", y_true.shape)
print("y_pred shape:", y_pred.shape)
print("Sample y_true:", y_true[:10])
print("Sample y_pred:", y_pred[:10])

## 5. Classification report

In [ ]:
report = save_classification_report(y_true, y_pred, class_names)

## 6. Confusion matrix

In [ ]:
save_confusion_matrix(y_true, y_pred, class_names)

## 7. Error analysis — top 10 most confused classes

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

# zero out diagonal (correct predictions) to find most confused pairs
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)

# get top 10 confused pairs
confused_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if cm_no_diag[i, j] > 0:
            confused_pairs.append((cm_no_diag[i, j], class_names[i], class_names[j]))

confused_pairs.sort(reverse=True)

print("Top 10 most confused class pairs (true → predicted):")
print(f"{'Count':>6}  {'True class':<30}  {'Predicted class'}")
print("-" * 65)
for count, true_cls, pred_cls in confused_pairs[:10]:
    print(f"{int(count):>6}  {true_cls:<30}  {pred_cls}")

## 8. Inference time

In [ ]:
import time

# warm up
for images, _ in test_ds.take(1):
    model.predict(images, verbose=0)

# measure over 100 batches
total_images = 0
start = time.time()

for images, _ in test_ds.take(100):
    model.predict(images, verbose=0)
    total_images += len(images)

elapsed = time.time() - start

ms_per_image = (elapsed / total_images) * 1000
fps = total_images / elapsed

print(f"Total images: {total_images}")
print(f"Total time:   {elapsed:.2f}s")
print(f"ms / image:   {ms_per_image:.2f} ms")
print(f"FPS:          {fps:.1f} images/sec")

## 9. Summary

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

macro_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
macro_recall    = recall_score(y_true, y_pred, average="macro", zero_division=0)
macro_f1        = f1_score(y_true, y_pred, average="macro", zero_division=0)
weighted_f1     = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("=" * 40)
print("CNN Baseline — Final Evaluation Summary")
print("=" * 40)
print(f"Test accuracy:      {test_accuracy * 100:.2f}%")
print(f"Test loss:          {test_loss:.4f}")
print(f"Macro precision:    {macro_precision:.4f}")
print(f"Macro recall:       {macro_recall:.4f}")
print(f"Macro F1-score:     {macro_f1:.4f}")
print(f"Weighted F1-score:  {weighted_f1:.4f}")
print(f"Inference speed:    {ms_per_image:.2f} ms/image  ({fps:.1f} FPS)")
print("=" * 40)